In [ ]:
# Install ReportLab if needed
!pip install reportlab

import pandas as pd
from google.colab import files
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
import numpy as np # Import numpy for np.nan

# ---- STEP 1: UPLOAD FILES ----
print("Upload the PITCHING spreadsheet (.xlsx):")
uploaded_pitch = files.upload()

print("Upload the CMJ spreadsheet (.xlsx):")
uploaded_cmj = files.upload()

pitch_file = list(uploaded_pitch.keys())[0]
cmj_file = list(uploaded_cmj.keys())[0]

# ---- STEP 2: LOAD DATA ----
pitch_df = pd.read_excel(pitch_file)
cmj_df   = pd.read_excel(cmj_file)

# ---- STEP 3: CLEAN COLUMN NAMES ----
# Remove leading/trailing spaces
pitch_df.columns = pitch_df.columns.str.strip()
cmj_df.columns = cmj_df.columns.str.strip()

# ---- STEP 4: CLEAN & STANDARDIZE DATES ----
pitch_df['Date'] = pd.to_datetime(pitch_df['Date'], errors='coerce')
cmj_df['Date'] = pd.to_datetime(cmj_df['Date'], errors='coerce')


# ---- STEP 5: FILTER GEO_PAT PITCHERS ----
Team_Name = "Your_Team_Name"

pitch_df = pitch_df[pitch_df['PitcherTeam'] == Team_Name]

# ---- STEP 6: DEFINE METRIC GROUPS ----
pitch_vars = [
    'RelSpeed', 'VertRelAngle', 'HorzRelAngle', 'SpinRate', 'SpinAxis', 'RelHeight', 'RelSide', 'Extension', 'VertBreak',
    'InducedVertBreak', 'HorzBreak'
]

cmj_vars = [
    'Concentric Duration',
    'Concentric Mean Force',
    'Concentric Mean Power / BM',
    'Concentric Peak Force',
    'Eccentric:Concentric Mean Force Ratio',
    'Eccentric Deceleration Phase Duration',
    'Eccentric Duration',
    'Eccentric Deceleration Mean Force',
    'Eccentric Peak Power',
    'Eccentric Peak Power / BW',
    'Eccentric Peak Velocity'

]

# ---- STEP 7: ONLY KEEP DATES PRESENT IN BOTH ----
valid_dates = set(pitch_df['Date']).intersection(set(cmj_df['Date']))
pitch_df = pitch_df[pitch_df['Date'].isin(valid_dates)]
cmj_df = cmj_df[cmj_df['Date'].isin(valid_dates)]

# ---- STEP 8: AVERAGE CMJ METRICS PER PITCHER + DATE + PITCH TYPE ----
cmj_avg = cmj_df.groupby(['Pitcher','Date','Pitch Type'])[cmj_vars].mean().reset_index()

# ---- STEP 9: MERGE PITCHING AND CMJ DATA ----
merged = pitch_df.merge(
    cmj_avg,
    left_on=['PitcherID','Date','TaggedPitchType'],
    right_on=['Pitcher','Date','Pitch Type'],
    suffixes=('_pitch','_cmj')
)

# ---- STEP 10: PREPARE PDF ----
output_path = "/content/correlation_results.pdf"
doc = SimpleDocTemplate(output_path)
elements = []
styles = getSampleStyleSheet()

# ---- STEP 11: COMPUTE CORRELATIONS ----
for pitcher_id in sorted(merged['PitcherID'].unique()):
    elements.append(Paragraph(f"Pitcher ID: {pitcher_id}", styles['Heading1']))
    pitcher_sub = merged[merged['PitcherID'] == pitcher_id]

    for ptype in sorted(pitcher_sub['TaggedPitchType'].unique()):
        elements.append(Paragraph(f"Pitch Type: {ptype}", styles['Heading2']))
        sub = pitcher_sub[pitcher_sub['TaggedPitchType'] == ptype]

        print(f"Pitcher {pitcher_id}, Pitch Type {ptype}, rows={len(sub)}")  # Debugging

        table_data = [['Pitch Metric', 'CMJ Metric', 'Correlation']]

        for p_var in pitch_vars:
            if p_var not in sub.columns:
                continue
            for c_var in cmj_vars:
                if c_var not in sub.columns:
                    continue

                # Remove NaNs but keep zeros
                valid_data = sub[[p_var, c_var]].dropna()
                if len(valid_data) < 2:
                    corr = 'N/A'
                else:
                    corr_val = valid_data[p_var].corr(valid_data[c_var])
                    corr = round(corr_val, 3) if pd.notnull(corr_val) else 'N/A'

                table_data.append([p_var, c_var, corr])

        table = Table(table_data, repeatRows=1)
        table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), colors.grey),
            ('GRID', (0,0), (-1,-1), 0.25, colors.black),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
            ('ALIGN', (0,0), (-1,-1), 'LEFT')
        ]))

        elements.append(table)
        elements.append(Spacer(1, 12))

    elements.append(PageBreak())

# ---- STEP 12: BUILD PDF ----
doc.build(elements)
print(f"PDF created: {output_path}")

# ---- STEP 13: DOWNLOAD PDF ----
files.download(output_path)